In [2]:
import pandas as pd

path = r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\mart\fact_contract_awards.csv"

df = pd.read_csv(path)

print(df.shape)
print(df.columns)

df.head()


(6196406, 5)
Index(['contract_id', 'award_date', 'supplier_name', 'num_offers',
       'award_year'],
      dtype='object')


,contract_id,award_date,supplier_name,num_offers,award_year
0,20192,21/12/18,The Executive Agency for Small and Medium-size...,2.0,2018
1,20193,21/12/18,The Executive Agency for Small and Medium-size...,1.0,2018
2,20211,23/12/20,European Union Agency for Cybersecurity,5.0,2020
3,20211,23/12/20,European Union Agency for Cybersecurity,NaN,2020
4,20211,23/12/20,European Union Agency for Cybersecurity,11.0,2020


In [3]:
date = pd.read_csv(r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\mart\dim_date.csv")

supplier = pd.read_csv(r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\mart\dim_supplier.csv")

print(date.columns)
print(supplier.columns)


Index(['date_id', 'award_date', 'year', 'quarter', 'month'], dtype='object')
Index(['supplier_id', 'supplier_name'], dtype='object')


In [4]:
# PASSO 1 — Profilazione rapida della fact table e parsing controllato di award_date.
# Scopo: verificare tipi, missing, duplicati e trasformare award_date in datetime in modo affidabile (day-first).
# Motivo: senza date correttamente parsate, qualsiasi trend per mese/quarter può risultare sbagliato.

import pandas as pd

path_fact = r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\mart\fact_contract_awards.csv"
df = pd.read_csv(path_fact)

# 1) Snapshot qualità: tipi e valori mancanti
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)

missing = df.isna().mean().sort_values(ascending=False) * 100
print("\nMissing % per colonna:\n", missing.round(2))

# 2) Duplicati (a livello di riga intera e per chiave logica minima)
print("\nDuplicati (riga intera):", df.duplicated().sum())
print("Duplicati per (contract_id, award_date, supplier_name):",
      df.duplicated(subset=["contract_id", "award_date", "supplier_name"]).sum())

# 3) Parsing data (dayfirst=True) + controllo errori di parsing
df["award_date_parsed"] = pd.to_datetime(df["award_date"], dayfirst=True, errors="coerce")

print("\nDate parse - valori non parsabili:", df["award_date_parsed"].isna().sum())
print("Range date (parsed):", df["award_date_parsed"].min(), "→", df["award_date_parsed"].max())

# 4) Coerenza anno: confronto tra award_year e anno derivato dalla data parsata
df["award_year_from_date"] = df["award_date_parsed"].dt.year
year_mismatch = (df["award_year_from_date"].notna()) & (df["award_year_from_date"] != df["award_year"])

print("\nMismatch anno (award_year vs anno da data):", year_mismatch.sum())


Shape: (6196406, 5)

Dtypes:
 contract_id        int64
award_date        object
supplier_name     object
num_offers       float64
award_year         int64
dtype: object

Missing % per colonna:
 num_offers       31.67
contract_id       0.00
award_date        0.00
supplier_name     0.00
award_year        0.00
dtype: float64

Duplicati (riga intera): 3833245
Duplicati per (contract_id, award_date, supplier_name): 4536523


C:\Users\karin\AppData\Local\Temp\ipykernel_30036\1312067279.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["award_date_parsed"] = pd.to_datetime(df["award_date"], dayfirst=True, errors="coerce")



Date parse - valori non parsabili: 0
Range date (parsed): 2018-01-01 00:00:00 → 2023-12-27 00:00:00

Mismatch anno (award_year vs anno da data): 0


In [5]:
# PASSO 2 — Validazione integrità temporale e identificazione duplicati strutturali
# Scopo: verificare che le date siano parsabili, coerenti con award_year e che non esistano duplicati logici.
# Motivo: duplicati o incoerenze temporali portano a KPI falsati (errore grave in contesto professionale).

# Parsing data con controllo rigoroso
df["award_date_parsed"] = pd.to_datetime(
    df["award_date"],
    dayfirst=True,
    errors="coerce"
)

# Conteggio date non parsabili
print("Date non parsabili:", df["award_date_parsed"].isna().sum())

# Range temporale reale del dataset
print("Data minima:", df["award_date_parsed"].min())
print("Data massima:", df["award_date_parsed"].max())

# Confronto anno derivato dalla data vs award_year
df["year_from_date"] = df["award_date_parsed"].dt.year

mismatch = df[df["year_from_date"] != df["award_year"]]

print("Mismatch anno:", len(mismatch))

# Verifica duplicati logici (stessa aggiudicazione)
dup = df.duplicated(
    subset=["contract_id", "supplier_name", "award_date"],
    keep=False
)

print("Righe duplicate logiche:", dup.sum())


C:\Users\karin\AppData\Local\Temp\ipykernel_30036\1119575265.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["award_date_parsed"] = pd.to_datetime(


Date non parsabili: 0
Data minima: 2018-01-01 00:00:00
Data massima: 2023-12-27 00:00:00
Mismatch anno: 0
Righe duplicate logiche: 5065323


In [6]:
# PASSO 3 — Rendere il parsing delle date deterministico e creare una versione deduplicata della fact.
# Scopo: eliminare il warning (formato esplicito) e produrre una fact "unica" a livello logico
#        (contract_id + supplier_name + award_date) per analisi aggregate affidabili.
# Nota: non si elimina la tabella originale; si crea df_dedup per KPI, mantenendo tracciabilità.

import pandas as pd

# 1) Parsing deterministico: dd/mm/yy (come nei tuoi esempi: 21/12/18)
df["award_date_parsed"] = pd.to_datetime(
    df["award_date"],
    format="%d/%m/%y",
    errors="coerce"
)

print("Date non parsabili (con format esplicito):", df["award_date_parsed"].isna().sum())

# 2) Deduplicazione logica per granularità minima
keys = ["contract_id", "supplier_name", "award_date_parsed"]

df_dedup = df.drop_duplicates(subset=keys, keep="first").copy()

print("Righe originali:", len(df))
print("Righe deduplicate:", len(df_dedup))
print("Righe rimosse:", len(df) - len(df_dedup))

# 3) Controllo rapido: quanti record per contract_id restano dopo dedup (distribuzione)
counts = df_dedup["contract_id"].value_counts()
print("\nContract_id - min righe:", counts.min())
print("Contract_id - max righe:", counts.max())
print("Contract_id - mediana righe:", counts.median())


Date non parsabili (con format esplicito): 0
Righe originali: 6196406
Righe deduplicate: 1659883
Righe rimosse: 4536523

Contract_id - min righe: 1
Contract_id - max righe: 1
Contract_id - mediana righe: 1.0


In [7]:
# PASSO 4 — Creare colonne temporali analitiche derivate dalla data.
# Scopo: rendere il dataset compatibile con analisi temporali standard BI (trend annuali, mensili, trimestrali).
# Motivo: le dashboard professionali usano sempre colonne anno, mese, trimestre, non solo la data grezza.

df_dedup["year"] = df_dedup["award_date_parsed"].dt.year
df_dedup["month"] = df_dedup["award_date_parsed"].dt.month
df_dedup["quarter"] = df_dedup["award_date_parsed"].dt.quarter

# versione leggibile per dashboard
df_dedup["year_month"] = df_dedup["award_date_parsed"].dt.to_period("M").astype(str)

# verifica
print(df_dedup[[
    "contract_id",
    "award_date_parsed",
    "year",
    "quarter",
    "month",
    "year_month"
]].head())


   contract_id award_date_parsed  year  quarter  month year_month
0        20192        2018-12-21  2018        4     12    2018-12
1        20193        2018-12-21  2018        4     12    2018-12
2        20211        2020-12-23  2020        4     12    2020-12
5        20212        2020-12-28  2020        4     12    2020-12
8        20216        2020-12-23  2020        4     12    2020-12


In [8]:
# PASSO 5 — Calcolare il numero di contratti per anno.
# Scopo: costruire il primo KPI base e verificare la distribuzione temporale del dataset.
# Motivo: il conteggio dei contratti per anno è il KPI più fondamentale nel procurement analytics
#        ed è il primo controllo per individuare anomalie o trend.

contracts_per_year = (
    df_dedup
    .groupby("year")
    .agg(num_contracts=("contract_id", "count"))
    .reset_index()
    .sort_values("year")
)

print(contracts_per_year)


   year  num_contracts
0  2018         236939
1  2019         264957
2  2020         268129
3  2021         286514
4  2022         301362
5  2023         301982


In [9]:
# PASSO 6 — Calcolare la crescita percentuale anno su anno (YoY).
# Scopo: misurare come evolve il numero di contratti nel tempo.
# Motivo: la crescita YoY è uno degli indicatori più utilizzati in analisi business e reporting executive.

contracts_per_year["prev_year_contracts"] = contracts_per_year["num_contracts"].shift(1)

contracts_per_year["yoy_growth_percent"] = (
    (contracts_per_year["num_contracts"] - contracts_per_year["prev_year_contracts"])
    / contracts_per_year["prev_year_contracts"]
) * 100

# Arrotondamento per leggibilità
contracts_per_year["yoy_growth_percent"] = contracts_per_year["yoy_growth_percent"].round(2)

print(contracts_per_year)


   year  num_contracts  prev_year_contracts  yoy_growth_percent
0  2018         236939                  NaN                 NaN
1  2019         264957             236939.0               11.82
2  2020         268129             264957.0                1.20
3  2021         286514             268129.0                6.86
4  2022         301362             286514.0                5.18
5  2023         301982             301362.0                0.21


In [11]:
# PASSO 7 — Esportare il dataset pulito e deduplicato in un nuovo file CSV.
# Scopo: creare un dataset "analytics-ready" da usare in Power BI.
# Motivo: Power BI deve lavorare su dati già puliti per garantire KPI corretti e performance elevate.

output_path = r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\processed\fact_contract_awards_clean.csv"

# Esportazione
df_dedup.to_csv(output_path, index=False)

print("File esportato in:", output_path)
print("Numero righe esportate:", len(df_dedup))


File esportato in: C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\processed\fact_contract_awards_clean.csv
Numero righe esportate: 1659883


In [12]:
# PASSO 13 — Arricchire la fact aggiungendo supplier_id tramite join con dim_supplier.
# Scopo: ottenere la chiave dimensionale corretta per costruire uno star schema valido.
# Motivo: le relazioni nei modelli dimensionali devono usare chiavi surrogate (supplier_id), non nomi.

# caricare dim_supplier
dim_supplier = pd.read_csv(
    r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\mart\dim_supplier.csv"
)

# join per ottenere supplier_id
df_enriched = df_dedup.merge(
    dim_supplier,
    on="supplier_name",
    how="left"
)

# controllo qualità join
print("Righe fact:", len(df_dedup))
print("Righe enriched:", len(df_enriched))
print("supplier_id mancanti:", df_enriched["supplier_id"].isna().sum())


Righe fact: 1659883
Righe enriched: 1659883
supplier_id mancanti: 0


In [13]:
# PASSO 14 — Verificare se esistono supplier_name associati a più supplier_id.
# Scopo: garantire che la chiave ottenuta sia stabile (1 nome -> 1 id).
# Motivo: se un nome mappa a più id, il modello dimensionale può diventare ambiguo.

ambiguous = (
    dim_supplier
    .groupby("supplier_name")["supplier_id"]
    .nunique()
    .reset_index(name="n_supplier_id")
)

ambiguous = ambiguous[ambiguous["n_supplier_id"] > 1]

print("supplier_name con più supplier_id:", len(ambiguous))

# Se esistono casi ambigui, mostriamo i primi 10
if len(ambiguous) > 0:
    display(ambiguous.head(10))


supplier_name con più supplier_id: 0


In [14]:
# PASSO 15 — Sovrascrivere il dataset clean con la versione enriched (con supplier_id).
# Scopo: permettere a Power BI di aggiornarsi con Refresh senza dover ricaricare il file.

output_path = r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\processed\fact_contract_awards_clean.csv"

df_enriched.to_csv(output_path, index=False)

print("File aggiornato con supplier_id")
print("Numero righe:", len(df_enriched))


File aggiornato con supplier_id
Numero righe: 1659883


In [15]:
import pandas as pd

df_level = pd.read_csv(r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\mart\fact_contract_level.csv")

print(df_level.columns)


Index(['contract_id', 'date_id', 'award_year', 'supplier_id', 'num_offers'], dtype='object')


In [16]:
import pandas as pd

df_staging = pd.read_csv(
    r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\staging\can_awards_staging_clean.csv"
)

print(df_staging.columns.tolist())


['ID_NOTICE_CAN', 'DT_DISPATCH', 'CAE_NAME', 'NUMBER_OFFERS', 'award_year']


In [17]:
# Importiamo la libreria pandas, necessaria per leggere e analizzare file CSV
import pandas as pd

# Definiamo il percorso completo del file fact_contract_awards_fk.csv
# Questo file è una fact table derivata e potrebbe contenere campi aggiuntivi
path_fk = r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\mart\fact_contract_awards_fk.csv"

# Carichiamo il file CSV in un DataFrame pandas
# Il DataFrame è la struttura dati principale per l'analisi tabellare in Python
df_fk = pd.read_csv(path_fk)

# Stampiamo la lista completa delle colonne presenti nel dataset
# Questo serve per verificare se esiste una colonna con il valore economico del contratto
print("Colonne presenti in fact_contract_awards_fk.csv:")
print(df_fk.columns.tolist())


Colonne presenti in fact_contract_awards_fk.csv:
['contract_id', 'award_date', 'award_year', 'supplier_id', 'num_offers']


In [18]:
# Importiamo pandas per lavorare con i file CSV
import pandas as pd

# Definiamo il percorso del file fact_contract_awards_fk2.csv
# Questo è un'altra fact table che potrebbe contenere campi aggiuntivi
path_fk2 = r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\mart\fact_contract_awards_fk2.csv"

# Carichiamo il file in un DataFrame
df_fk2 = pd.read_csv(path_fk2)

# Stampiamo la lista delle colonne per verificare la presenza del valore economico
print("Colonne presenti in fact_contract_awards_fk2.csv:")
print(df_fk2.columns.tolist())


Colonne presenti in fact_contract_awards_fk2.csv:
['contract_id', 'date_id', 'award_year', 'supplier_id', 'num_offers']


In [19]:
# Importiamo pandas per leggere il file CSV
import pandas as pd

# Definiamo il percorso del file fact_contract_awards.csv
# Questo è il file fact principale e potrebbe contenere più informazioni
path_awards = r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\mart\fact_contract_awards.csv"

# Carichiamo il file in un DataFrame pandas
df_awards = pd.read_csv(path_awards)

# Stampiamo tutte le colonne presenti per verificare se esiste il valore economico
print("Colonne presenti in fact_contract_awards.csv:")
print(df_awards.columns.tolist())


Colonne presenti in fact_contract_awards.csv:
['contract_id', 'award_date', 'supplier_name', 'num_offers', 'award_year']


In [20]:
# Importiamo pandas per leggere il file CSV
import pandas as pd

# Definiamo il percorso del file raw originale TED
# Questo file contiene i dati completi prima della trasformazione
path_raw = r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\raw\TED - Contract award notices 2018 - 2023\export_CAN_2023_2018.csv"

# Carichiamo solo le prime 1000 righe per velocità (il file è molto grande)
df_raw = pd.read_csv(path_raw, nrows=1000)

# Stampiamo tutte le colonne disponibili
# Qui verifichiamo se esiste il valore economico del contratto
print("Colonne presenti nel file raw TED:")
print(df_raw.columns.tolist())


Colonne presenti nel file raw TED:
['ID_NOTICE_CAN', 'TED_NOTICE_URL', 'YEAR', 'ID_TYPE', 'DT_DISPATCH', 'XSD_VERSION', 'CANCELLED', 'CORRECTIONS', 'B_MULTIPLE_CAE', 'CAE_NAME', 'CAE_NATIONALID', 'CAE_ADDRESS', 'CAE_TOWN', 'CAE_POSTAL_CODE', 'CAE_GPA_ANNEX', 'ISO_COUNTRY_CODE', 'ISO_COUNTRY_CODE_GPA', 'B_MULTIPLE_COUNTRY', 'ISO_COUNTRY_CODE_ALL', 'CAE_TYPE', 'EU_INST_CODE', 'MAIN_ACTIVITY', 'B_ON_BEHALF', 'B_INVOLVES_JOINT_PROCUREMENT', 'B_AWARDED_BY_CENTRAL_BODY', 'TYPE_OF_CONTRACT', 'TAL_LOCATION_NUTS', 'B_FRA_AGREEMENT', 'FRA_ESTIMATED', 'B_FRA_CONTRACT', 'B_DYN_PURCH_SYST', 'CPV', 'MAIN_CPV_CODE_GPA', 'ID_LOT', 'ADDITIONAL_CPVS', 'B_GPA', 'GPA_COVERAGE', 'LOTS_NUMBER', 'VALUE_EURO', 'VALUE_EURO_FIN_1', 'VALUE_EURO_FIN_2', 'B_EU_FUNDS', 'TOP_TYPE', 'B_ACCELERATED', 'OUT_OF_DIRECTIVES', 'CRIT_CODE', 'CRIT_PRICE_WEIGHT', 'CRIT_CRITERIA', 'CRIT_WEIGHTS', 'B_ELECTRONIC_AUCTION', 'NUMBER_AWARDS', 'ID_AWARD', 'ID_LOT_AWARDED', 'INFO_ON_NON_AWARD', 'INFO_UNPUBLISHED', 'B_AWARDED_TO_A_GROUP

In [21]:
# Verifichiamo la qualità del campo valore contratto

# Carichiamo solo le colonne necessarie per velocità
df_value = pd.read_csv(
    path_raw,
    usecols=["ID_NOTICE_CAN", "AWARD_VALUE_EURO"],
    nrows=100000
)

# Convertiamo in numerico (gestisce eventuali errori)
df_value["AWARD_VALUE_EURO"] = pd.to_numeric(
    df_value["AWARD_VALUE_EURO"],
    errors="coerce"
)

# Stampiamo statistiche base
print("Numero totale righe:", len(df_value))
print("Valori non null:", df_value["AWARD_VALUE_EURO"].notna().sum())
print("Percentuale null:", df_value["AWARD_VALUE_EURO"].isna().mean()*100)

print("\nStatistiche:")
print(df_value["AWARD_VALUE_EURO"].describe())


Numero totale righe: 100000
Valori non null: 71723
Percentuale null: 28.277

Statistiche:
count    7.172300e+04
mean     2.209163e+06
std      5.210249e+07
min      0.000000e+00
25%      3.907175e+03
50%      3.973750e+04
75%      2.571972e+05
max      1.000000e+10
Name: AWARD_VALUE_EURO, dtype: float64


In [22]:
# Importiamo pandas
import pandas as pd

# Percorso file raw
path_raw = r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\raw\TED - Contract award notices 2018 - 2023\export_CAN_2023_2018.csv"

# Carichiamo solo le colonne necessarie
df_value = pd.read_csv(
    path_raw,
    usecols=["ID_NOTICE_CAN", "AWARD_VALUE_EURO"]
)

# Rinominiamo per coerenza con il modello esistente
df_value = df_value.rename(columns={
    "ID_NOTICE_CAN": "contract_id",
    "AWARD_VALUE_EURO": "award_value_euro"
})

# Convertiamo in numerico
df_value["award_value_euro"] = pd.to_numeric(
    df_value["award_value_euro"],
    errors="coerce"
)

# Rimuoviamo valori null
df_value = df_value.dropna()

# Rimuoviamo duplicati
df_value = df_value.drop_duplicates(subset=["contract_id"])

# Salviamo il file
output_path = r"C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\mart\fact_contract_value.csv"

df_value.to_csv(output_path, index=False)

print("File creato:", output_path)
print("Numero righe:", len(df_value))


File creato: C:\Users\karin\Portfolio\progetto2\eu-procurement-analytics\data\mart\fact_contract_value.csv
Numero righe: 1437562
